<a href="https://colab.research.google.com/github/gautamkr1876/AIML_ClassNotes/blob/main/8.%20Agentic%20AI%20Systems/19.%20Difussion%20Models%20Fundamentals/L18_%20Diffusion_Models_Lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Diffusion Models: From Blurry VAEs to Stable Diffusion

Over the past decade, generative AI has evolved rapidly. We started with models like Variational Autoencoders (VAEs), which were stable but often produced blurry images. Then came Generative Adversarial Networks (GANs), capable of generating incredibly realistic images—but at the cost of unstable training and limited diversity.

For years, researchers believed this trade-off was unavoidable.

Then diffusion models introduced a completely different way of thinking about image generation. Instead of trying to generate an image in a single step, they learned to gradually remove noise from random pixels, eventually reconstructing a realistic image.

Today, this simple idea powers systems like Stable Diffusion, Midjourney, Adobe Firefly, and many modern image generation models.

In this notebook, we'll build an intuitive understanding of diffusion models and learn how they evolved.

## Roadmap

1. The Generative Trilemma – Why VAEs and GANs weren't enough
2. A New Paradigm: Generation as Gradual Refinement
3. The Forward Diffusion Process
4. The Reverse Diffusion Process
5. How Diffusion Models Learn (Training Objective)
6. DDPM: Putting Everything Together
7. Why DDPM Needed Improvements
8. DDIM: Faster Sampling
9. Classifier-Free Guidance
10. Latent Diffusion and Stable Diffusion
11. Wrap-Up and Key Takeaways

# Section 1 — The Generative Trilemma: Why Diffusion Models Were Needed

Before we understand **how diffusion models work**, it's worth asking a more fundamental question:

> **Why were diffusion models invented in the first place?**

For many years, researchers had two major approaches for generating images. Each solved some problems remarkably well—but each came with significant trade-offs.

This led to what many people informally refer to as the **Generative Trilemma**:

> Can a generative model simultaneously provide:
>
> - High-quality images
> - High diversity (generate many different outputs)
> - Stable and reliable training

For a long time, the answer seemed to be **no**.

Let's see why.

---

## 1.1 Variational Autoencoders (VAEs): The Stable Compressor

Variational Autoencoders (VAEs) take an image, compress it into a compact **latent representation**, and then reconstruct the image from that latent vector.

Instead of memorizing every training image, the encoder learns a smooth latent space that follows a simple distribution (typically a standard Gaussian). Because of this regularized latent space, we can sample entirely new latent vectors and decode them into new images.

One clever idea that makes VAEs train efficiently is the **reparameterization trick**.

Instead of sampling directly,

$$
z \sim N(\mu, \sigma^2)
$$
we rewrite the sampling process as

```python
z = μ + σ × ε
```

where

```python
ε ~ N(0,1)
```

This small reformulation allows gradients to flow normally during backpropagation, making the entire model differentiable.

### Strengths

- Stable and easy to train
- Excellent coverage of the data distribution
- Smooth latent space useful for interpolation and representation learning

### Weakness

The reconstruction objective (typically Mean Squared Error) encourages the decoder to average over many plausible outputs.

Instead of committing to one sharp solution, it predicts the average of several possibilities.

The result?

> Images are usually **blurry**.

---

## 1.2 GANs: The Realistic Artist

Generative Adversarial Networks (GANs) take a completely different approach.

Instead of reconstructing images, two neural networks compete against each other:

- A **Generator** tries to create realistic fake images.
- A **Discriminator** tries to distinguish real images from generated ones.

Both networks improve together in a minimax game.

When this adversarial training succeeds, GANs produce remarkably realistic images—often with a single forward pass.

### Strengths

- Extremely sharp images
- Very fast inference
- High visual realism

### Weaknesses

However, GANs introduced a new set of problems.

- **Mode Collapse:** The generator discovers a few outputs that consistently fool the discriminator and keeps producing variations of those images instead of learning the full data distribution.

- **Training Instability:** Since two networks are learning simultaneously, balancing their progress is difficult. If one becomes too strong, training may oscillate or completely fail.

- **No Explicit Likelihood:** Measuring how well the generator actually models the underlying data distribution is difficult.

---

## 1.3 The Generative Trilemma

Notice the pattern.

VAEs are easy to train and generate diverse samples, but their outputs lack visual quality.

GANs generate beautiful images, but training them reliably is difficult, and they often sacrifice diversity.

| Property | VAE | GAN |
|----------|-----|-----|
| Training Stability | ✅ High | ❌ Low |
| Image Quality | ⚠️ Moderate (blurry) | ✅ Excellent |
| Diversity | ✅ High | ❌ Can suffer from mode collapse |

Researchers wanted something that could achieve all three:

- Stable training
- High-quality images
- High diversity

This seemed impossible...

until diffusion models introduced a completely different way of thinking about image generation.

---

Researchers wanted a model that could finally achieve all three:

- Stable training
- High-quality image generation
- High sample diversity

Diffusion models turned out to be that breakthrough.

But they didn't achieve it by making GANs or VAEs slightly better.

Instead, they introduced a completely different way of thinking about image generation.

Before we dive into the mechanics of diffusion models, let's first understand this new perspective and why it makes the seemingly impossible possible.

---
## Quiz 1 — Why Were Diffusion Models Needed?

### Question

A startup is evaluating two image generation models.

- **Model A** consistently produces highly realistic images, but its training is unstable and it often generates only a few similar-looking outputs.
- **Model B** trains reliably and produces diverse images, but its outputs are noticeably blurry.

The team now wants a model that combines the strengths of both approaches.

**Which statement best explains why diffusion models became popular?**

**A.** They generate images faster than both GANs and VAEs.

**B.** They combine stable training with high-quality and diverse image generation by learning to gradually remove noise.

**C.** They eliminate the need for latent representations altogether.

**D.** They replace neural networks with deterministic mathematical equations.

### Correct Answer

**B.** They combine stable training with high-quality and diverse image generation by learning to gradually remove noise.

### Explanation

Diffusion models were introduced because they provide **stable training**, **high-quality image generation**, and **strong sample diversity**, overcoming many of the limitations of both VAEs and GANs.


## Section 2 — A New Paradigm: Generation as Gradual Refinement

In the previous section, we saw why neither **VAEs** nor **GANs** could simultaneously achieve stable training, high-quality images, and strong diversity.

This raises an important question:

> **What if we stopped trying to generate an entire image in one giant leap?**

That simple question led researchers to a completely different way of thinking about image generation.

Instead of asking a neural network to solve one extremely difficult problem—

> **Random Noise → Realistic Image**

diffusion models divide it into thousands of much smaller problems:

> **Slightly Noisy Image → Slightly Cleaner Image**

Rather than making one perfect prediction, the model continuously refines its output through many small improvements.

This is the central idea behind diffusion models and is known as **generation through gradual refinement**.

---

### 2.1 Why Is This Easier?

Imagine you're restoring an old painting covered with years of dust and grime.

You wouldn't expect a conservator to restore the entire painting with one brush stroke.

Instead, they would carefully remove a very thin layer of dirt, inspect the result, remove another thin layer, and continue until the original artwork slowly reappears.

Diffusion models follow exactly the same philosophy.

Each denoising step is deliberately kept small.

Instead of learning

> **Noise → Perfect Image**

the model repeatedly learns

> **Very Noisy Image → Slightly Less Noisy Image**

Since every prediction is much simpler, the learning process becomes significantly more stable.

Thousands of these tiny improvements eventually reconstruct a realistic image.

---

### 2.2 The Two Processes

Diffusion models consist of two complementary processes.

The first process **gradually destroys** an image by adding small amounts of random noise.

The second process **gradually reconstructs** an image by learning to remove that noise.

```
Forward Process                    Reverse Process

Clean Image                        Pure Noise
      │                                 │
      ▼                                 ▲
Add a little noise              Remove a little noise
      │                                 ▲
      ▼                                 ▲
Add a little more noise         Remove a little more noise
      │                                 ▲
      ▼                                 ▲
Eventually → Pure Noise      Eventually → Clean Image
```

We'll study each of these processes separately.

We'll begin with the **forward diffusion process**, where we intentionally corrupt an image until it becomes pure random noise.

---
## Quiz 2 — Generation as Gradual Refinement

### Question

Two researchers are discussing how to build an image generation model.

- **Researcher A** wants a neural network to generate a complete image directly from random noise in a single prediction.
- **Researcher B** proposes breaking the task into thousands of small denoising steps, where each step only removes a tiny amount of noise.

Why is Researcher B's approach the key idea behind diffusion models?

**A.** Because neural networks cannot process random noise directly.

**B.** Because solving many small denoising problems is much easier to learn than generating an entire image in a single step.

**C.** Because Gaussian noise must always be removed gradually.

**D.** Because using multiple steps eliminates the need for neural networks.

### Correct Answer

**B.** Because solving many small denoising problems is much easier to learn than generating an entire image in a single step.

### Explanation

The central idea behind diffusion models is **generation through gradual refinement**.

Breaking image generation into thousands of simple denoising tasks makes the learning problem much easier and more stable.


# Section 3 — The Forward Diffusion Process

In the previous section, we learned that diffusion models generate images by **gradually removing noise** instead of generating everything in a single step.

But before a model can learn to remove noise, we first need a systematic way to **add** it.

This process is called the **forward diffusion process**.

The forward process is **not learned**. It is a fixed procedure whose only purpose is to gradually destroy the information present in an image.

Once we understand how an image is corrupted, we'll later learn how a neural network reverses this process to generate entirely new images.

---

### 3.1 Intuition Behind the Forward Process

Imagine dropping a drop of ink into a glass of clear water.

Initially, the ink has a well-defined shape.

As time passes, it spreads throughout the water.

Eventually, the ink becomes so evenly distributed that there's no way to tell where it originally started.

Another way to think about this is a printed document left outside during a dust storm.

At first, every word is perfectly readable.

As more dust accumulates, the letters become harder to recognize.

Eventually, the page is completely covered, and all useful information is lost.

The forward diffusion process works in exactly the same way.

We begin with a clean image and repeatedly add a tiny amount of random Gaussian noise until the image becomes indistinguishable from pure random noise.

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/724/original/image_%287%29.png?1785761461" height=500>

---

### 3.2 What is Gaussian Noise?

Gaussian noise is simply random noise sampled from a **normal (Gaussian) distribution**.

Instead of changing the entire image dramatically, we slightly perturb every pixel by adding a small random value.

Visually:

- A small amount of noise looks similar to film grain.
- More noise gradually hides important details.
- Enough noise eventually turns the image into television-like static, where no recognizable structure remains.

The key point is that every step introduces **only a tiny amount** of randomness.

---

### 3.3 Corrupting an Image Step by Step

Rather than destroying the image all at once, diffusion models corrupt it gradually over many timesteps.

```
x₀ (Clean Image)
      ↓
x₁ (Slightly Noisy)
      ↓
x₂
      ↓
⋮
      ↓
xₜ⁄₂ (Image still recognizable)
      ↓
⋮
      ↓
xₜ (Pure Noise)
```

Notice that no individual step drastically changes the image.

Each step introduces only a very small amount of additional noise.

This gradual corruption is what makes the reverse process possible.

---

### 3.4 Why Not Add All the Noise at Once?

Suppose we skipped all the intermediate steps and directly transformed

```
Clean Image → Pure Noise
```

Now imagine asking a neural network to reverse that process in a single prediction.

It would have to solve:

> **Pure Noise → Realistic Image**

This is an extremely difficult problem because almost all information about the original image has been destroyed.

Instead, diffusion models create thousands of tiny intermediate states.

Now the model only has to solve a much easier problem:

> **Given a slightly noisy image, can you remove just the small amount of noise that was recently added?**

Each prediction becomes much simpler.

When these simple predictions are chained together, they gradually reconstruct the entire image.

This is the central idea behind diffusion models.

---

### 3.5 The Noise Schedule

A natural question now is:

> **How do we decide how much noise to add at each step?**

This is controlled by the **noise schedule**.

The noise schedule is simply a predefined sequence that specifies the amount of Gaussian noise added during every timestep.

Some commonly used schedules are:

- **Linear Schedule** – the amount of noise increases at a constant rate. This was used in the original DDPM paper.

- **Cosine Schedule** – the noise increases more slowly near the beginning and the end while increasing faster in the middle. This preserves useful image information for longer and often produces better image quality.

An important point is that the noise schedule is **fixed**.

It is **not learned** by the model.

The model only learns how to reverse the corruption produced by this schedule.

---

At this point, we understand how a clean image is gradually transformed into pure noise.

The obvious next question is:

> **If adding noise is easy, how can a neural network learn to reverse this process and generate entirely new images?**

## Section 4 — The Reverse Diffusion Process

In the previous section, we saw how a clean image can be gradually corrupted into pure random noise through a fixed forward process.

Now comes the more interesting question:

> **If adding noise is easy, can we learn to reverse the entire process?**

The answer is yes—and that is exactly what diffusion models are trained to do.

Instead of generating an image in a single step, they learn to reverse the forward process one tiny step at a time.

---

### 4.1 Walking the Same Path Backwards

Recall what the forward process looked like.
Generation simply follows the same path in reverse.

```
Pure Noise
      ↓
Remove a little noise
      ↓
Remove a little more noise
      ↓
⋮
      ↓
Clean Image
```

Notice something important.

The model is **not** trying to magically convert random noise into a perfect image in one prediction.

Instead, every step only makes the image **slightly cleaner** than it was before.

Thousands of these tiny improvements eventually produce a realistic image.

---

### 4.2 Why the Reverse Process Is Learnable

Think back to the painting restoration analogy.

Suppose you're restoring a centuries-old painting covered in layers of dirt.

You wouldn't try to clean the entire painting in one stroke.

Instead, you would remove one thin layer of dirt, inspect the result, remove another layer, and continue until the original artwork slowly reappears.

Reverse diffusion works in exactly the same way.

Each denoising step is intentionally small.

Because every step only needs to make a tiny correction, the learning problem becomes much simpler than generating an entire image all at once.

This idea is what makes diffusion models both stable and effective.

---

### 4.3 Image Generation Starts from Pure Noise

Unlike traditional image editing, diffusion models don't begin with an existing image.

Instead, they start with **pure Gaussian noise**.

At the beginning, the image looks completely random.

After a few denoising steps, faint structures begin to emerge.

As more steps are performed, these structures gradually become recognizable objects.

The final denoising step produces a realistic image.

---

### 4.4 The Same Neural Network at Every Step

A natural question is:

> **Who decides what should be removed at each step?**

Diffusion models use a neural network—typically a **U-Net**—to guide every denoising step.

An interesting design choice is that we do **not** train a different neural network for every timestep.

Instead, the **same U-Net** is reused throughout the entire reverse process.

The only thing that changes is the current timestep.

This tells the network how noisy the current image is, allowing it to make the appropriate correction.

Using one shared network makes the model much more efficient while allowing it to perform the complete denoising process.

---

### 4.5 Putting the Reverse Process Together

At this point, we understand the overall idea behind image generation.

However, one important question still remains.

Notice that we never explained **what the U-Net is actually trained to predict** during each denoising step.

Does it predict the clean image?

Does it predict the next image?

Or does it predict something else entirely?

That question leads us to the training objective used in modern diffusion models.

---
## Quiz 3 — Understanding the Reverse Diffusion Process

### Question

A student says:

> "If diffusion models eventually generate a clean image, why don't they simply train a neural network that converts pure random noise directly into the final image in one step?"

Which response best explains why diffusion models instead use many iterative denoising steps?

**A.** Neural networks cannot process pure random noise as input.

**B.** Breaking generation into many small denoising steps transforms a difficult problem into a sequence of much simpler learning tasks.

**C.** Gaussian noise can only be removed one timestep at a time.

**D.** Every denoising step uses a completely different neural network.

### Correct Answer

**B.** Breaking generation into many small denoising steps transforms a difficult problem into a sequence of much simpler learning tasks.

### Explanation

Diffusion models succeed because they avoid solving the extremely difficult task of generating an entire image in a single prediction.

Instead, they repeatedly perform small denoising steps, each making only a minor correction to the current image.
This gradual refinement strategy is the core reason diffusion models are stable and effective.


## Section 5 — How Diffusion Models Learn (Training Objective)

In the previous section, we saw that a **U-Net** is responsible for gradually removing noise during the reverse diffusion process.

But this raises an important question:

> **What exactly should the U-Net learn to predict?**

At first glance, the answer seems obvious.

Why not simply train the network to predict the original clean image?

Surprisingly, that is **not** what modern diffusion models do.

Instead, they learn something much simpler.

---

### 5.1 Predicting the Noise Instead of the Image

Suppose we have a noisy image `xₜ`.

One possible training strategy would be:

```
Noisy Image
      │
      ▼
Predict Clean Image
```

This sounds intuitive, but in practice it turns out to be surprisingly difficult.

Instead, diffusion models train the network to do something else:

```
Noisy Image
      │
      ▼
Predict the Noise that was Added
```

Once we know the noise present in the image, we can remove it and obtain a cleaner image.

This simple change makes training significantly more stable.

---

### 5.2 Why Is Predicting Noise Easier?

Imagine someone walks into your room and dumps a pile of papers onto your desk.

Now they ask you:

> **"What did your desk look like five minutes ago?"**

There are countless possible answers.

Your books could have been arranged differently.

Your notebook might have been open or closed.

Your laptop could have been anywhere.

Reconstructing the entire original scene is difficult.

Now consider a different question.

> **"Which papers were just added?"**

That is much easier.

You only need to identify the **new information** rather than reconstruct everything that already existed.

Diffusion models use exactly this idea.

Instead of asking the network to imagine an entire clean image, they simply ask it to identify the noise that was recently added.

Once that noise is known, removing it becomes straightforward.

---

### 5.3 How the Model Is Trained

During training, we already have access to clean images from our dataset.

For every training iteration, we perform the following steps:

1. Start with a real image `x₀`.
2. Randomly choose a timestep `t`.
3. Add the appropriate amount of Gaussian noise to obtain `xₜ`.
4. Keep a copy of the exact noise that was added.
5. Give the noisy image `xₜ` and the timestep `t` to the U-Net.
6. Ask the U-Net to predict the noise.
7. Compare the predicted noise with the actual noise.

If the prediction is inaccurate, the model updates its weights through backpropagation.

After repeating this process over millions of images, the U-Net gradually becomes very good at estimating the noise present at any timestep.

---

### 5.4 The Training Loss

The training objective is surprisingly simple.

We compare:

- The **actual noise** that was added.
- The **noise predicted** by the U-Net.

The difference between the two is measured using **Mean Squared Error (MSE)**.

The simplified DDPM training objective is:

$$
L_{\text{simple}}
=
\mathbb{E}_{x_0,\epsilon,t}
\left[
\left\|
\epsilon -
\epsilon_\theta(x_t,t)
\right\|^2
\right]
$$

Conceptually, this is nothing more than:

> **"Predict the exact noise that was added."**

The closer the prediction is to the true noise, the smaller the loss becomes, and the better the model learns to reverse the diffusion process.

---

### 5.5 The Complete Training Pipeline

Putting everything together, one training iteration looks like this:

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/711/original/image_%281%29.png?1785760914" height=500>

Notice something interesting.

The model is **never shown the reverse diffusion process directly**.

It only learns to predict the noise present in randomly sampled noisy images.

Yet, after training, repeatedly applying this simple denoising operation allows the model to generate entirely new images from pure noise.

---

We've now seen all the individual building blocks of diffusion models:

- The forward diffusion process
- The reverse diffusion process
- The training objective

The next step is to see how all of these components come together in the **Denoising Diffusion Probabilistic Model (DDPM)**, the first diffusion model that demonstrated this complete framework.

---
## Quiz 4 — How Diffusion Models Learn

### Question

During training, why do modern diffusion models usually ask the U-Net to predict the **noise** added to an image instead of directly predicting the original clean image?

**A.** Because identifying the added noise is a simpler and more stable learning problem than reconstructing the entire image directly.

**B.** Because the original clean image is unknown during training.

**C.** Because predicting the clean image requires a different U-Net for every timestep.

**D.** Because Gaussian noise cannot be removed mathematically.

### Correct Answer

**A.** Because identifying the added noise is a simpler and more stable learning problem than reconstructing the entire image directly.

### Explanation

During training, the original clean image is already available, so the model can compare its predictions against the exact noise that was added.
Predicting the added noise is a simpler regression problem and leads to more stable and effective training.


## Section 6 — DDPM: Putting Everything Together

So far, we've studied the individual pieces of diffusion models separately.

- We learned how a clean image is gradually corrupted through the **forward diffusion process**.
- We learned how the **reverse diffusion process** reconstructs an image from pure noise.
- We also saw how the model is trained by predicting the noise present at each timestep.

Now it's time to combine all of these ideas into one complete framework.

This framework is called the **Denoising Diffusion Probabilistic Model (DDPM)**, introduced in 2020.

---

### 6.1 The Big Picture

DDPM consists of two phases.

**Training**

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/719/original/image_%286%29.png?1785761011" height=500>

After repeating this process millions of times, the U-Net learns how to estimate the noise present at every timestep.

---

**Inference (Image Generation)**

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/713/original/image_%282%29.png?1785760942" height=500>

Notice the difference.

During **training**, we already have real images.

During **inference**, we don't.

Instead, we begin with random noise and repeatedly apply the learned denoising process.

---

### 6.2 Why Is a U-Net Used?

At this point, we know that a neural network is responsible for predicting the noise present in an image.

But what kind of neural network should we use?

The answer is the **U-Net**.

Originally developed for biomedical image segmentation, the U-Net is particularly well suited for diffusion models because it needs to understand both the **global structure** of an image and its **fine local details**.

Its architecture consists of three main parts.

---

#### Encoder (Downsampling Path)

The encoder gradually reduces the spatial resolution of the image.

As the image becomes smaller, the network learns increasingly high-level features.

For example,

```
512 × 512

↓

256 × 256

↓

128 × 128

↓

64 × 64
```

At these lower resolutions, the network captures the overall structure of the scene, such as the positions of objects and their relationships.

---

#### Bottleneck

At the lowest resolution, the network has a compressed representation containing the most important semantic information about the image.

This is where the model has the largest receptive field and can reason about the image as a whole.

---

#### Decoder (Upsampling Path)

The decoder gradually reconstructs the spatial resolution.

```
64 × 64

↓

128 × 128

↓

256 × 256

↓

512 × 512
```

During this process, the network predicts the noise present in the image while recovering fine visual details.

---

#### Skip Connections

One challenge with deep networks is that high-resolution details can be lost during downsampling.

To solve this, the U-Net introduces **skip connections**.

These connections directly pass feature maps from the encoder to the decoder at matching resolutions.

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/714/original/image_%283%29.png?1785760959" height=500>

This allows the decoder to recover sharp edges, textures, and small details that might otherwise be lost.

---

#### Time Embeddings

Unlike a traditional U-Net, the diffusion U-Net also receives the current timestep.

This timestep is converted into a **time embedding**, which tells the network **how noisy the current image is**.

For example,

- Early timesteps correspond to images that are almost pure noise.
- Later timesteps correspond to images that require only slight denoising.

The time embedding enables the same U-Net to perform the correct denoising operation throughout the entire reverse diffusion process.

---

### 6.3 Why Does the U-Net Need the Timestep?

The same U-Net is used for every denoising step.

But denoising an image at timestep **900** is very different from denoising one at timestep **50**.

At early timesteps, the image is almost pure noise.

Near the end, only a tiny amount of noise remains.

Therefore, the network also receives the current timestep as an additional input.

This timestep is converted into a **time embedding** (often using sinusoidal embeddings) and injected into multiple layers of the U-Net.

This tells the network **how noisy the current image is**, allowing it to apply the correct amount of denoising.

---

### 6.4 The Mathematics of DDPM

We've built the intuition for DDPM already. Here we add just enough math to make it precise — three equations, each with the idea behind it.

---

#### 6.4.1 The Forward Process: Step by Step, Then the Shortcut

Each single forward step just adds a bit of noise to the previous image:

$$
x_1 = \sqrt{1-\beta_1}\ x_0 + \sqrt{\beta_1}\ \epsilon_1
$$

$$
x_2 = \sqrt{1-\beta_2}\ x_1 + \sqrt{\beta_2}\ \epsilon_2
$$

...and so on, up to $x_T$. Each $\epsilon_i \sim N(0, I)$ is a fresh, independent sample of random noise.

**Reading the symbols:**

- $x_0$ — the original clean image you started with.
- $x_1$ — the image after one small dose of noise has been added; $x_2$ after two doses, and so on.
- $\beta_1, \beta_2, \dots$ — small numbers (like 0.0001) that control *how much* noise gets added at each step. Think of them as a dial: bigger $\beta$, more noise added that step.
- $\epsilon_1, \epsilon_2, \dots$ — the actual random noise added at each step (freshly sampled every time, like rolling dice again at each step).
- $\sqrt{1-\beta_1}$ — a shrinking factor applied to the image before adding noise, so pixel values don't blow up as this repeats hundreds of times.

So the equation $x_1 = \sqrt{1-\beta_1}\ x_0 + \sqrt{\beta_1}\ \epsilon_1$ reads as: *"take the original image, shrink it slightly, and add a small dash of random noise — that's the image one step noisier."*

**Intuition:** $x_2$ is built from $x_1$, which is itself built from $x_0$ — noise is compounding on top of noise, step after step. If you substitute the $x_1$ equation into the $x_2$ equation, you get $x_2$ written directly in terms of $x_0$ and a combination of $\epsilon_1, \epsilon_2$. Do that substitution all the way to $x_T$, and (because sums of Gaussians are still Gaussian) the whole chain collapses into one clean equation that jumps straight from $x_0$ to any $x_t$:

$$
x_t = \sqrt{\bar\alpha_t}\ x_0 + \sqrt{1-\bar\alpha_t}\ \epsilon, \qquad \epsilon \sim N(0, I)
$$

Here $\bar\alpha_t$ is just a number between 0 and 1 that depends on the noise schedule and how far along we are in the process ($\bar\alpha_t \approx 1$ near the start, $\bar\alpha_t \approx 0$ near the end).

**Reading the symbols:**

- $x_t$ — the noisy image at *any* step $t$ you want to jump to directly (no need to go through $x_1, x_2, \dots$ one at a time).
- $\bar\alpha_t$ ("alpha-bar") — a single number summarizing how much of the *original image* survives by step $t$. Close to 1 means barely touched; close to 0 means almost fully replaced by noise.
- $\sqrt{\bar\alpha_t}\ x_0$ — the "how much original image is left" portion.
- $\sqrt{1-\bar\alpha_t}\ \epsilon$ — the "how much noise has been mixed in" portion, using one single fresh noise sample $\epsilon$.
- $t$ — just a label for which timestep we're looking at (an integer, e.g. between 1 and 1000).

So this equation reads as: *"the noisy image at step $t$ is a weighted mix of the clean image and pure noise — how much of each depends only on $t$."*

**Intuition:** the noisy image is just a blend of the original image and random noise — mostly image early on, mostly noise late on. This one line is why training is cheap: we can jump to any noisy step in one shot instead of simulating hundreds of small steps.

---

#### 6.4.2 What the U-Net Actually Predicts

Rather than predicting the denoised image directly, the U-Net is trained to predict the **noise** $\epsilon$ that was added — call the prediction $\epsilon_\theta(x_t, t)$.

**Reading the symbols:** $\epsilon_\theta(x_t, t)$ just means "the noise estimate that the neural network (with learned weights $\theta$) outputs, when you show it the noisy image $x_t$ and tell it which timestep $t$ it's looking at." It's the network's *guess* at $\epsilon$ — not the true $\epsilon$ itself.

**Intuition:** predicting "what noise is in here" is an easier, more consistent target for the network than predicting "what the clean image looks like," which changes drastically depending on the timestep. This is why DDPM works well in practice.

---

#### 6.4.3 The Training Loss

$$
L_{\text{simple}}(\theta) = \mathbb{E}_{t,\ x_0,\ \epsilon}\left[\ \left\| \epsilon - \epsilon_\theta(x_t, t) \right\|^2\ \right]
$$

**Reading the symbols:**

- $L_{\text{simple}}(\theta)$ — the loss: a single number that tells us how wrong the network currently is (lower is better). $\theta$ just means "as a function of the network's current weights."
- $\mathbb{E}_{t,\ x_0,\ \epsilon}[\ \cdot\ ]$ — "average this over many random draws of $t$, $x_0$, and $\epsilon$." In practice this just means: repeat the process below many times over many training examples and average the result.
- $\epsilon$ — the *true* noise we actually added (we know this, because we added it ourselves).
- $\epsilon_\theta(x_t, t)$ — the network's *guess* at that noise.
- $\|\ \cdot\ \|^2$ — squared distance between two things. Here, between the true noise and the guessed noise. Small when the guess is close, large when it's far off.

So the whole line reads as: *"on average, how far off is the network's noise guess from the real noise?"* — and training just means nudging $\theta$ to make that number smaller.

In words: pick a real image, pick a random timestep, add noise using the formula from 6.4.1, ask the U-Net to guess the noise, and penalize it for how wrong the guess is (mean squared error). That's the entire training loop from Section 6.1, written as one line.

---

#### 6.4.4 Sampling: Reversing the Process, Step by Step

The reverse process runs the forward process in the opposite direction. Starting from pure noise $x_T$, it removes noise step by step to recover $x_0$. One step looks like:

$$
p_\theta(x_{t-1} \mid x_t) = N\left(x_{t-1};\ \mu_\theta(x_t, t),\ \sigma_t^2 I\right)
$$

**Reading the symbols:**

- $p_\theta(x_{t-1} \mid x_t)$ — "the probability distribution over what the slightly-less-noisy image $x_{t-1}$ could be, *given that* we're currently looking at $x_t$." The $\theta$ reminds us this comes from our trained network.
- $N(\cdot\ ;\ \text{mean},\ \text{variance})$ — a Gaussian (bell-curve) distribution, described by where it's centered (mean) and how spread out it is (variance).
- $\mu_\theta(x_t, t)$ — the *mean* of that bell curve: the network's best single guess for $x_{t-1}$.
- $\sigma_t^2 I$ — the *spread* around that guess: a small, fixed amount of variance (not learned), the same in every direction ($I$ is just "apply this equally across all pixels").

So this equation reads as: *"the model doesn't output one exact image for $x_{t-1}$ — it outputs a best guess ($\mu_\theta$), plus a small allowed wobble ($\sigma_t^2$) around that guess."*

Applied repeatedly, this is what the reverse chain looks like:

$$
p_\theta(x_{T-1} \mid x_T) = N\left(x_{T-1};\ \mu_\theta(x_T, T),\ \sigma_T^2 I\right)
$$

$$
p_\theta(x_{T-2} \mid x_{T-1}) = N\left(x_{T-2};\ \mu_\theta(x_{T-1}, T-1),\ \sigma_{T-1}^2 I\right)
$$

...and so on, down to $x_0$.

Here, $\mu_\theta$ is the mean predicted by the neural network. At each step, the model takes the current noisy state and the timestep as input, and predicts the mean of the previous, slightly-less-noisy state.

**The key question: what exactly should the model learn to predict?** There are three options:

- Predict $x_0$ directly (the original data)
- Predict $\mu_\theta$ directly (the mean of the reverse step)
- Predict $\epsilon$ (the added noise itself)

DDPM picks the third option — predicting $\epsilon$, as established in 6.4.2.

#### 6.4.5 Putting It Together: What Training and Sampling Actually Do

Zoom out for a second — everything we've built across 6.4.1 to 6.4.4 boils down to just two loops, and together they are DDPM. Here's what each one is actually doing, in plain terms.

**Training:**

Take a real image. Pick a random point on the noise timeline — anywhere from barely-touched to almost-pure-noise. Corrupt the image to that exact level using the one-shot formula from 6.4.1. Show the corrupted image (and which timestep it's at) to the U-Net, and ask it to guess what noise is hiding inside. Compare its guess to the noise you actually added, and nudge the network's weights so next time it guesses a little better. Now repeat that — grab another real image, pick another random noise level — over and over, millions of times, across the whole dataset, until the network's guesses stop improving.

That's it. That's the entire training process. There's no separate "step 1, step 2, step 3" to memorize — it's the same single move (corrupt → guess the noise → correct the guess) repeated endlessly on random images at random noise levels, until the U-Net has effectively seen every stage of noise, on every kind of image, enough times to get good at all of them.

**Sampling(Denoising):**

Start with an image that's nothing but random static — there's no real content in it at all yet. Show it to the trained U-Net and ask: "what noise do you see in here?" Use that answer to peel away a small amount of noise, landing on a slightly cleaner image. Now do the exact same thing again with this new image — ask the U-Net what noise it sees, peel a little more away. Keep repeating this, walking backward through the noise levels one at a time, all the way from "pure static" down to "final image." At every step except the very last one, a tiny bit of fresh randomness gets mixed back in on purpose — that's what makes the model capable of generating different images from different starting noise, instead of always producing the exact same picture. On the final step, that extra randomness is switched off, so the finished image doesn't come out with leftover noise sitting on top of it.

That's the whole generation process: repeatedly asking a trained denoiser "what noise is this?" and removing a little at a time, starting from static and ending on a picture — which is also exactly why it's slow: there's no way to jump ahead, only to take the next step.

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/217/677/original/image.png?1786361462" height=500>


### 6.5 The Next Challenge

DDPM solved one of the biggest problems in generative AI:

- Stable training
- High-quality images
- Strong diversity

However, it introduced a new challenge:

> **Image generation was simply too slow for real-world applications.**

Researchers now faced a different question.

> **Can we generate equally good images using far fewer denoising steps?**

This question led directly to the development of **DDIM (Denoising Diffusion Implicit Models)**, which we'll study next.

 ## Section 7 — The Three Challenges of DDPM

At this point, DDPM seems like the ideal generative model.

It offers:

- Stable training
- High-quality image generation
- Excellent sample diversity

So, is the problem solved?

Not quite.

Although DDPM was a huge breakthrough, researchers quickly discovered that it still had several practical limitations.

These limitations motivated the next generation of diffusion models.

Let's look at the three biggest challenges.

---

### 7.1 Challenge 1 — Slow Image Generation

Recall how DDPM generates an image.

```
Pure Noise
     │
     ▼
Step T
     │
     ▼
Step T-1
     │
     ▼
Step T-2
     │
     ▼
...
     │
     ▼
Step 2
     │
     ▼
Step 1
     │
     ▼
Generated Image
```

Each denoising step depends on the output of the previous step.

This means the reverse process is **sequential**.

Unlike many deep learning models, we cannot compute multiple denoising steps in parallel.

A typical DDPM performs around **1000 denoising steps**.

Since each step requires one forward pass through the U-Net, generating even a single image can take several seconds.

For research, this was acceptable.

For real-world applications, however, it was far too slow.

This naturally raised an important question:

> **Can we generate equally good images using far fewer denoising steps?**

We'll answer that in the next section with **DDIM**.

---

### 7.2 Challenge 2 — Lack of Control

Another limitation of the original DDPM is that it is **unconditional**.

Suppose we start from random noise.

What kind of image will the model generate?

A cat?

A mountain?

A car?

A portrait?

We have no control.

The model simply generates a random sample from the data distribution it learned during training.

In practical applications, however, users rarely want random images.

Instead, they want to specify exactly what should be generated.

For example:

- "A golden retriever playing in the snow."
- "A watercolor painting of a castle at sunset."
- "A futuristic city at night."

This leads to another question:

> **How can we guide the diffusion process using a text prompt?**

We'll explore this later when we discuss **Classifier-Free Guidance**.

---

### 7.3 Challenge 3 — Pixel-Space Computation

The original DDPM performs the entire diffusion process directly on image pixels.

For a high-resolution image such as **1024 × 1024**, that means the U-Net must repeatedly process over a million pixel values at every denoising step.

Now combine that with roughly **1000 denoising iterations**.

The computational cost becomes enormous.

This makes training and inference expensive in terms of both memory and computation.

Researchers therefore began asking another question:

> **What if we performed diffusion in a much smaller latent space instead of directly on pixels?**

That idea eventually became **Latent Diffusion**, which powers Stable Diffusion.

---

### 7.4 Looking Ahead

Interestingly, each limitation of DDPM inspired a different breakthrough.

| Challenge | Solution |
|-----------|----------|
| Slow sampling | **DDIM** |
| Lack of controllability | **Classifier-Free Guidance** |
| Expensive pixel-space computation | **Latent Diffusion Models** |

Instead of replacing DDPM completely, researchers improved it step by step.

Each innovation solved one specific limitation while preserving the strengths of diffusion models.

We'll begin by tackling the first challenge:

> **How can we generate images much faster without sacrificing quality?**

## Section 8 — DDIM: Faster Sampling Without Retraining

In the previous section, we saw that the biggest practical limitation of DDPM is its **slow sampling process**.

Although each denoising step is simple, generating an image typically requires around **1000 sequential U-Net forward passes**.

This naturally raises an important question:

> **Do we really need every single denoising step?**

Researchers discovered that the answer is surprisingly **no**.

This idea led to **DDIM (Denoising Diffusion Implicit Models)**, which dramatically speeds up image generation without changing how the model is trained.

---

### 8.1 The Core Idea Behind DDIM

Recall the reverse diffusion process in DDPM.

```
1000
 │
 ▼
999
 │
 ▼
998
 │
 ▼
...
 │
 ▼
2
 │
 ▼
1
 │
 ▼
Generated Image
```

Every timestep must be visited.

The next step cannot begin until the previous one has finished.

DDIM asks a simple question:

> **What if we skipped some of these intermediate steps?**

Instead of visiting every timestep, DDIM can follow a much shorter path.

```
1000
 │
 ▼
950
 │
 ▼
900
 │
 ▼
800
 │
 ▼
650
 │
 ▼
...
 │
 ▼
0
```

Rather than taking tiny steps throughout the entire trajectory, DDIM makes larger jumps while still following the same overall denoising direction.

This allows images to be generated much faster.

---

### 8.2 Why Is This Possible?

The key insight behind DDIM is that the reverse process does not always have to be random.

DDPM introduces a small amount of randomness during every denoising step.

DDIM instead follows a **deterministic trajectory** from noise to image.

This means that, given:

- the same initial random noise
- the same prompt
- the same sampling settings

DDIM will always produce the same final image.

This deterministic behavior also makes image editing and inversion possible, topics that later became important in many diffusion-based editing systems.

---
### 8.3 The Mathematics of DDIM

Just enough math to see *why* skipping steps is even mathematically valid.

---

#### 8.3.1 The Trick: Estimate $x_0$, Then Jump

Recall from 6.4.1 that any noisy image can be written directly in terms of the clean image and a noise term:

$$
x_t = \sqrt{\bar\alpha_t}\ x_0 + \sqrt{1-\bar\alpha_t}\ \epsilon
$$

In DDPM, $\epsilon$ was a fresh random sample at every step. DDIM treats it differently: here, $\epsilon$ is the U-Net's own prediction $\epsilon_\theta(x_t, t)$ — a fixed, deterministic value once $x_t$ and $t$ are given, not a new random draw. That shift is what makes everything below deterministic.

Since the U-Net already predicts $\epsilon$, we can rearrange this equation to get an estimate of the clean image at any step:

$$
\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\ \epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}
$$

**Reading the symbols:**

- $\hat{x}_0$ — the "hat" means *estimate*: this is the network's guess at what the fully clean image looks like, made from wherever we currently are in the noisy chain.
- $x_t$ — the noisy image we're currently holding, at whatever timestep $t$ we happen to be at.
- $\epsilon_\theta(x_t, t)$ — the network's noise guess for this image (same notation as in the DDPM section).
- $\sqrt{1-\bar\alpha_t}$ and $\sqrt{\bar\alpha_t}$ — the same "how much noise vs. how much image" scaling factors from 6.4.1, just being used here to undo the mixing instead of create it.

This is really just the DDPM shortcut equation from 6.4.1, algebraically flipped around: instead of *"clean image + noise → noisy image,"* it says *"noisy image − predicted noise → estimated clean image."*

**Intuition:** instead of only ever asking "what does the *next* step look like?", the model can now guess the *entire clean image* in one shot, using whatever noisy step it's currently looking at.

---

#### 8.3.2 Jumping to Any Earlier Step — Not Just the Next One

Once we have $\hat{x}_0$, we plug it back into the *same* noising formula, but targeting **any** earlier timestep $s$ (not just $t-1$):

$$
x_s = \sqrt{\bar\alpha_s}\ \hat{x}_0 + \sqrt{1-\bar\alpha_s}\ \epsilon_\theta(x_t, t)
$$

**Reading the symbols:**

- $s$ — *any* earlier timestep we want to land on, not necessarily $t-1$. This is the whole trick: $s$ could be $t-1$, or $t-50$, or anything smaller than $t$.
- $x_s$ — the image we get after jumping straight to timestep $s$.
- $\hat{x}_0$ — the clean-image estimate from the step above; it doesn't change no matter which $s$ we're aiming for.
- $\bar\alpha_s$ — the same "how much original image survives" number from 6.4.1, but now evaluated at our *target* step $s$ instead of the current step $t$.
- $\epsilon_\theta(x_t, t)$ — notice this stays the same noise prediction from the current step $x_t$; we're not asking the network anything new, just reusing its one guess to land wherever we like.

So this equation reads as: *"take the estimated clean image, and re-noise it — but only by the amount appropriate for timestep $s$, whatever $s$ we choose."*

Two steps of this, skipping ahead each time:

$$
x_{950} = \sqrt{\bar\alpha_{950}}\ \hat{x}_0 + \sqrt{1-\bar\alpha_{950}}\ \epsilon_\theta(x_{1000}, 1000)
$$

$$
x_{900} = \sqrt{\bar\alpha_{900}}\ \hat{x}_0 + \sqrt{1-\bar\alpha_{900}}\ \epsilon_\theta(x_{950}, 950)
$$

...and so on, down to $x_0$.

**Intuition:** this is the entire reason skipping works. In DDPM, each step only knew how to produce *the very next* timestep. Here, because we route through an explicit estimate $\hat{x}_0$, the same formula works for *any* target timestep $s < t$ — so nothing stops us from jumping from 1000 straight to 950, or 900, instead of crawling through every single step in between.

---

#### 8.3.3 Where the Determinism Comes From

Setting the noise term to zero at every step ($\sigma_t = 0$) makes the whole trajectory deterministic — the same starting noise always produces the same image, which is what makes DDIM sampling reproducible and invertible, as noted in 8.2.

**Reading the symbols:** $\sigma_t$ is the same "allowed wobble" variable from the DDPM reverse-process equation ($\sigma_t^2 I$ in 6.4.4) — it controls how much fresh randomness gets injected at each step. In DDPM, $\sigma_t$ is always some positive number, so there's always a bit of new randomness added. In DDIM, we simply choose to set $\sigma_t = 0$ — no fresh randomness at all — so every step becomes a fixed, repeatable calculation rather than a fresh dice roll.

DDPM, by contrast, always injects fresh randomness ($\sigma_t z$) at every step, which is exactly why it can't be run this way.

### 8.4 Faster Sampling Without Retraining

One of the most elegant aspects of DDIM is that **nothing changes during training**.

The U-Net is trained exactly as before.

It still learns to predict the noise present in an image.

The only difference is **how we use that trained model during inference**.

This means:

- Same training process
- Same trained U-Net
- Different sampling procedure

In other words, DDIM is **not a new diffusion model**.

It is simply a more efficient way of sampling from a model that has already been trained.

---

### 8.5 Speed vs. Quality

Skipping denoising steps naturally introduces a trade-off.

| Number of Sampling Steps | Speed | Image Quality |
|--------------------------|-------|---------------|
| ~1000 | Slow | Highest |
| ~50 | Much Faster | Nearly identical in many cases |
| ~10–20 | Extremely Fast | Slightly lower quality |

In practice, many modern diffusion pipelines use between **20 and 50 sampling steps**, producing images that are visually very close to those generated using the full DDPM process while being dramatically faster.

---

### 8.6 DDPM vs. DDIM

| DDPM | DDIM |
|------|------|
| Visits every timestep | Can skip timesteps |
| Stochastic sampling | Deterministic sampling |
| Around 1000 sampling steps | Often 20–50 sampling steps |
| Slower inference | Much faster inference |
| Original sampling algorithm | Alternative sampling algorithm using the same trained model |

The important takeaway is that **DDIM improves inference speed without changing the training process**.

---

DDIM solved one major challenge of diffusion models:

> **Slow image generation.**

However, another important limitation still remained.

Even with faster sampling, diffusion models still generated **random** images.

Users couldn't simply type:

> "A futuristic city at sunset"

and expect the model to generate exactly that.

The next breakthrough solved this problem by allowing diffusion models to follow **text prompts** during image generation.

## Section 9 — Classifier-Free Guidance: Teaching Diffusion Models to Follow Instructions

In the previous section, we solved one major limitation of diffusion models.

DDIM showed that we could generate images much faster.

However, another important problem still remained.

Suppose we start from random noise.

What kind of image should the model generate?

A cat?

A mountain?

A sports car?

A castle?

Without any additional information, the model simply generates a **random image** from the distribution it learned during training.

But in real-world applications, users don't want random images.

They want to describe exactly what they want.

For example,

> "A futuristic city at sunset"

or

> "A watercolor painting of a panda reading a book"

The question now becomes:

> **How can we guide the diffusion process using a text prompt?**

---

### 9.1 From Unconditional to Conditional Generation

The original DDPM is an **unconditional** generative model.

```
Random Noise
      │
      ▼
Generated Image
```

The generated image is completely random.

To make image generation useful, we need to condition the model on additional information.

For text-to-image generation, that additional information is simply the user's prompt.

```
Random Noise + Text Prompt
             │
             ▼
       Generated Image
```

Now the model is no longer generating random images.

It is generating images that match the prompt.

---

### 9.2 The Original Solution: Classifier Guidance

Early diffusion models achieved this using a separate image classifier.

The process looked like this:

```
Text Prompt
      │
      ▼
Image Classifier
      │
      ▼
Guide the Diffusion Model
```

The classifier continuously evaluated whether the generated image matched the desired category and nudged the denoising process in the correct direction.

Although effective, this approach had several drawbacks.

- It required training a completely separate classifier.
- The classifier had to be available during inference.
- Training and deployment became more complicated.

Researchers wondered:

> **Can the diffusion model learn to guide itself?**

The answer was **Classifier-Free Guidance**.

---

### 9.3 The Key Idea Behind Classifier-Free Guidance

Instead of training an additional classifier, we modify how the diffusion model itself is trained.

During training, every text prompt is handled in one of two ways.

Most of the time, the model receives the correct prompt.

```
Image + "A red sports car"
```

But occasionally, the prompt is deliberately removed.

```
Image + Empty Prompt
```

This teaches the same U-Net to perform two different tasks.

- **Conditional prediction** — generate an image that matches the prompt.
- **Unconditional prediction** — generate an image without any prompt.

Since the same model learns both behaviors, no external classifier is needed.

---

### 9.4 How Guidance Works During Inference

During image generation, we run the U-Net twice.

**First prediction**

```
Random Noise
+
Text Prompt

↓

Conditional Prediction
```

**Second prediction**

```
Random Noise
+
Empty Prompt

↓

Unconditional Prediction
```

We then combine these two predictions.

Conceptually,

```
Final Prediction

=

Unconditional Prediction

+

Guidance Strength ×
(Conditional Prediction − Unconditional Prediction)
```

The unconditional prediction represents what the model would naturally generate.

The conditional prediction represents what the prompt is asking for.

The difference between them tells us how to steer the generation toward the prompt.

---

### 9.5 Understanding the Guidance Scale

The **guidance scale** controls how strongly the model follows the prompt.

| Guidance Scale | Behavior |
|---------------|----------|
| 1 | Very little guidance; images may ignore the prompt. |
| 5–8 | Good balance between creativity and prompt adherence. |
| 10–15 | Strongly follows the prompt but may appear less natural. |
| Very High (20+) | Can introduce artifacts or over-constrained images. |

This is why you'll often see code such as:

```python
guidance_scale = 7.5
```

This value provides a good balance between image quality and prompt fidelity for many Stable Diffusion models.

---

### 9.6 Why Classifier-Free Guidance Was a Breakthrough

Classifier-Free Guidance solved one of the biggest practical problems in diffusion models.

It allowed image generation to be:

- Prompt-controlled
- Simpler to train
- Simpler to deploy
- Compatible with the same diffusion model

This idea became one of the key building blocks behind modern text-to-image systems such as Stable Diffusion.

---

At this point, our diffusion model is:

- Stable
- High quality
- Fast (using DDIM)
- Controllable through text prompts

One challenge still remains.

Even with faster sampling, every denoising step still operates directly on millions of image pixels.

Can we make the diffusion process itself much more computationally efficient?

That question leads us to **Latent Diffusion**, the foundation of Stable Diffusion.

## Section 10 — Latent Diffusion and Stable Diffusion

So far, we've solved two major limitations of diffusion models.

- **DDIM** made image generation much faster.
- **Classifier-Free Guidance** allowed us to control image generation using text prompts.

However, one important challenge still remained.

Even with fewer denoising steps, the diffusion process was still operating directly on high-resolution images.

Researchers therefore asked another question:

> **Do we really need to perform diffusion on millions of image pixels?**

Surprisingly, the answer is **no**.

This idea led to **Latent Diffusion Models (LDMs)**, which form the foundation of **Stable Diffusion**.

---

### 10.1 The Pixel-Space Problem

Consider a 512 × 512 RGB image.

It contains

```
512 × 512 × 3 ≈ 786,000 values
```

During image generation, the U-Net processes these values repeatedly across multiple denoising steps.

Even if DDIM reduces the number of sampling steps, every step still requires processing a large image.

As image resolution increases, both memory usage and computation grow rapidly.

Researchers wanted to make each denoising step itself much cheaper.

---

### 10.2 A Familiar Idea: The VAE

At the beginning of this notebook, we explored **Variational Autoencoders (VAEs)**.

Although VAEs were not ideal standalone generative models, they were excellent at compressing images into compact latent representations.

Stable Diffusion takes advantage of this strength.

Instead of performing diffusion directly on the image, it first compresses the image using a VAE encoder.

For example,

```
512 × 512 × 3 Image
          │
          ▼
64 × 64 × 4 Latent Representation
```

The latent representation contains roughly **48× fewer values** than the original image.

This dramatically reduces the amount of computation required during every denoising step.

---

### 10.3 Latent Diffusion

Instead of performing diffusion in pixel space,

```
Image

↓

Forward Diffusion

↓

Reverse Diffusion

↓

Generated Image
```

Latent Diffusion performs the same process inside the compressed latent space.

```
Image
   │
   ▼
VAE Encoder
   │
   ▼
Latent Representation
   │
   ▼
Forward Diffusion
   │
   ▼
Reverse Diffusion
   │
   ▼
Clean Latent
   │
   ▼
VAE Decoder
   │
   ▼
Generated Image
```

Notice that the diffusion model **never directly processes the full-resolution image**.

The expensive iterative denoising happens entirely inside the latent space.

Only the encoder and decoder operate on pixels.

---

### 10.4 Putting Everything Together: Stable Diffusion

Stable Diffusion combines all of the concepts we've learned throughout this notebook into a single system.

#### Step 1 — Understand the Prompt

The user's prompt is converted into a numerical representation using the **CLIP Text Encoder**.

```
"A futuristic city at sunset"

↓

Text Embedding
```

This embedding captures the semantic meaning of the prompt.

---

#### Step 2 — Generate a Latent Image

Generation starts with **random latent noise**.

Using

- the latent representation
- the timestep
- the text embedding
- Classifier-Free Guidance

the U-Net gradually removes noise over multiple denoising steps.

Modern implementations typically use **DDIM or similar fast samplers** instead of the original DDPM sampler.

The result is a clean latent representation.

---

#### Step 3 — Decode the Latent

Finally, the **VAE Decoder** converts the clean latent back into a full-resolution image.

The complete Stable Diffusion pipeline is:

<img src="https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/215/717/original/image_%285%29.png?1785760990" height=500 width=400>
---

### 10.5 How Every Piece Fits Together

Looking back, every topic we've covered contributes to Stable Diffusion.

| Concept | Role in Stable Diffusion |
|---------|--------------------------|
| Forward Diffusion | Creates the training corruption process |
| Reverse Diffusion | Gradually generates an image |
| U-Net | Predicts the noise at every timestep |
| DDPM | Introduced the complete diffusion framework |
| DDIM | Speeds up image generation |
| Classifier-Free Guidance | Makes image generation follow text prompts |
| VAE | Compresses images into latent space |
| CLIP | Converts text prompts into embeddings |

Stable Diffusion is not a completely new model.

It is a carefully engineered system that combines several important innovations into one practical text-to-image pipeline.

This combination is what made modern AI image generation both efficient and widely accessible.

---

We've now completed the theoretical journey from basic generative models to Stable Diffusion.

The final step is to see these concepts in action by building a Stable Diffusion pipeline using the Hugging Face `diffusers` library.(in the next class)

## Section 11 — Wrap-Up and Key Takeaways

Throughout this notebook, we've explored the complete evolution of modern diffusion models—from understanding the limitations of GANs and VAEs to seeing how Stable Diffusion generates high-quality images.

We learned how the **forward diffusion process** gradually adds noise, how the **reverse diffusion process** learns to remove it, and why predicting the **added noise** makes training both simple and stable. We then brought these ideas together through **DDPM**, explored how **DDIM** speeds up sampling, how **Classifier-Free Guidance** enables prompt-controlled generation, and how **Latent Diffusion** makes the entire process computationally efficient.

The result is **Stable Diffusion**, which combines a **CLIP Text Encoder**, **U-Net**, and **VAE** to generate realistic images from text prompts.

### In the next class:-


- Stable Diffusion XL (SDXL) and Stable Diffusion 3 (SD3)
- Loading different checkpoints and using LoRA
- Prompt engineering and negative prompts
- Seeds for reproducible generation
- Different schedulers and sampling strategies
- Building complete image generation pipelines with the Hugging Face Diffusers library

**Thank you!**